In [52]:
import pandas as pd
import numpy as np

train = pd.read_csv(r'data/train.csv')
test  = pd.read_csv(r'data/test.csv')

print(f"Train: {train.shape} | Test: {test.shape}")

# what columns does test NOT have? those are what we predict
print("Missing from test:", [c for c in train.columns if c not in test.columns])

Train: (1200, 13) | Test: (120, 11)
Missing from test: ['emotional_state', 'intensity']


In [53]:
# quick sanity check
train.head(3)

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and concentra...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, yet my thou...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I feel more se...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3


In [54]:
train.dtypes

id                      int64
journal_text              str
ambience_type             str
duration_min            int64
sleep_hours           float64
energy_level            int64
stress_level            int64
time_of_day               str
previous_day_mood         str
face_emotion_hint         str
reflection_quality        str
emotional_state           str
intensity               int64
dtype: object

In [55]:
# face_emotion_hint has a lot missing — 10% gone, will need special handling
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
pd.DataFrame({'count': missing, '%': missing_pct})[missing > 0]

,count,%
sleep_hours,7,0.58
previous_day_mood,15,1.25
face_emotion_hint,123,10.25


In [56]:
# are the targets balanced? matters a lot for model choice
print(train['emotional_state'].value_counts())
print()
print(train['intensity'].value_counts().sort_index())
# both look pretty even — no class weighting needed

emotional_state
calm           216
restless       209
neutral        201
focused        193
mixed          191
overwhelmed    190
Name: count, dtype: int64

intensity
1    226
2    228
3    240
4    277
5    229
Name: count, dtype: int64


In [57]:
# categorical distributions — just eyeballing for now
for col in ['ambience_type', 'time_of_day', 'previous_day_mood',
            'face_emotion_hint', 'reflection_quality']:
    print(f"\n{col}:")
    print(train[col].value_counts(dropna=False))


ambience_type:
ambience_type
ocean       268
mountain    252
forest      231
cafe        229
rain        220
Name: count, dtype: int64

time_of_day:
time_of_day
afternoon        319
morning          302
night            286
evening          268
early_morning     25
Name: count, dtype: int64

previous_day_mood:
previous_day_mood
mixed          213
restless       208
neutral        203
overwhelmed    202
calm           182
focused        177
NaN             15
Name: count, dtype: int64

face_emotion_hint:
face_emotion_hint
neutral_face    214
calm_face       197
none            174
happy_face      173
tense_face      164
tired_face      155
NaN             123
Name: count, dtype: int64

reflection_quality:
reflection_quality
clear         411
vague         400
conflicted    389
Name: count, dtype: int64


In [58]:
train[['duration_min', 'sleep_hours', 'energy_level', 'stress_level']].describe().round(2)

,duration_min,sleep_hours,energy_level,stress_level
count,1200.00,1193.00,1200.00,1200.00
mean,15.86,5.99,3.02,3.03
std,7.67,1.50,1.38,1.40
min,3.00,3.50,1.00,1.00
25%,10.00,5.00,2.00,2.00
50%,15.00,6.00,3.00,3.00
75%,20.00,7.00,4.00,4.00
max,35.00,8.50,5.00,5.00


In [59]:
# word count is more useful than char length for short noisy texts
train['word_count'] = train['journal_text'].str.split().str.len()
print(train['word_count'].describe())

# how many entries are dangerously short?
short = train[train['word_count'] <= 5]
print(f"\nShort texts (<=5 words): {len(short)}")
short[['journal_text', 'emotional_state', 'intensity']].head(6)

count    1200.000000
mean       10.904167
std         5.278561
min         2.000000
25%         7.000000
50%        11.000000
75%        14.000000
max        32.000000
Name: word_count, dtype: float64

Short texts (<=5 words): 232


,journal_text,emotional_state,intensity
485,kinda calm now,overwhelmed,2
486,felt heavy,overwhelmed,3
487,back to normal after,restless,4
488,honestly okay session.,calm,1
491,For some reason okay session.,neutral,5
493,At first felt heavy.,overwhelmed,2


In [60]:
# does stress actually track with emotional_state? checking my assumption
print(train.groupby('emotional_state')['stress_level'].mean().round(2).sort_values(ascending=False))
print()
print(train.groupby('emotional_state')['energy_level'].mean().round(2).sort_values(ascending=False))
# focused has high energy but low stress — makes sense
# overwhelmed has high stress but avg energy — interesting

emotional_state
overwhelmed    3.17
neutral        3.15
restless       3.12
mixed          3.02
calm           2.90
focused        2.80
Name: stress_level, dtype: float64

emotional_state
focused        3.30
mixed          3.01
calm           2.98
overwhelmed    2.97
restless       2.95
neutral        2.91
Name: energy_level, dtype: float64


In [61]:
# curious: conflicted reflections — are they harder to classify?
# if yes, they might be noisy training samples
conflicted = train[train['reflection_quality'] == 'conflicted']
print(f"Conflicted rows: {len(conflicted)}")
print(conflicted['emotional_state'].value_counts())
print(f"avg stress in conflicted: {conflicted['stress_level'].mean():.2f}")
print(f"avg stress overall:       {train['stress_level'].mean():.2f}")

Conflicted rows: 389
emotional_state
calm           75
overwhelmed    66
focused        64
neutral        63
restless       63
mixed          58
Name: count, dtype: int64
avg stress in conflicted: 2.88
avg stress overall:       3.03


In [62]:
# another hunch: when face_emotion_hint is missing, what's the emotional state?
# if it's random, we can just impute. if it's systematic, we need a flag feature
missing_face = train[train['face_emotion_hint'].isnull()]
print(f"Rows with missing face hint: {len(missing_face)}")
print(missing_face['emotional_state'].value_counts())
print()
# compare to overall distribution — is any state overrepresented?
print("Overall distribution:")
print(train['emotional_state'].value_counts())

Rows with missing face hint: 123
emotional_state
calm           24
overwhelmed    22
focused        22
restless       21
mixed          19
neutral        15
Name: count, dtype: int64

Overall distribution:
emotional_state
calm           216
restless       209
neutral        201
focused        193
mixed          191
overwhelmed    190
Name: count, dtype: int64


In [63]:
# spot check — just reading some rows to see if the data makes intuitive sense
pd.set_option('display.max_colwidth', 80)
train[['journal_text', 'emotional_state', 'intensity',
       'stress_level', 'energy_level', 'word_count']].sample(8, random_state=7)

,journal_text,emotional_state,intensity,stress_level,energy_level,word_count
459,After a few minutes the tension eased a bit for a moment. ...,calm,1,3,2,13
457,During the session I couldn't sit still mentally and I just observed it. ...,restless,4,4,1,14
1022,after the session i felt distracted most of the time. i stayed with it anywa...,restless,5,5,1,16
141,I noticed it felt like a normal moment which surprised me.,neutral,5,2,3,11
668,honestly not much change,neutral,3,2,2,4
1011,not gonna lie i felt heavy in the chest. i was more tired than i thought. ...,overwhelmed,4,4,4,17
151,At first I kept thinking about other tasks so I sat with that feeling.,restless,3,4,2,14
830,"during the session felt lighter, not fully though. after some time that help...",calm,3,5,1,15
